preliminary experiment with llama

In [ ]:
import pandas as pd
import requests
import json
import time
import nltk

from nltk.corpus import framenet as fn
from nltk.corpus.reader.framenet import PrettyList
from nltk.tokenize import sent_tokenize
from groq import Groq

In [ ]:
frames = list()
for frame in PrettyList(fn.frames()):
    frame_str = str()
    frame_str += f"{frame.name}"
    # frame_str += f"Frame:\n{frame.name}"
    # frame_str += f"\nDefinition:\n{frame.definition}"
    # frame_str += f"\nFrame Elements:\n"
    # num_fes = 1
    # for fe in frame.FE.keys():
    #     frame_str += f"{num_fes}. {fe}\n"
    #     num_fes += 1
    frame_str += '\n'
    frames.append(frame_str)

len(frames)

In [ ]:
for frame in PrettyList(fn.frames()):
    print(frame.frameRelations)
    break

In [ ]:
print(''.join(frames))

In [ ]:
with open("corpora/en_ukwac/processed_ukwac.txt", 'r') as f:
    ukwac = f.readlines()

ukwac = ukwac[1:10]

In [ ]:
len(ukwac)

In [ ]:
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

In [ ]:
example = """For the text: "She runs fast."

Words and annotations might be:

[
  {"word": "She", "fee": false, "frame": "-"},
  {"word": "runs", "fee": true, "frame": "Self_motion"},
  {"word": "fast", "fee": false, "frame": "-"}
]
"""

In [ ]:
PROMPT = """TASK: Annotate the given TEXT word by word with Berkeley FrameNet frames from the provided FRAMES list.

INSTRUCTIONS:
1. Using Fillmore's Frame Semantics theory, inspect each word in the TEXT considering its context.
2. For every word, assign the parameters in JSON array format: 'word', 'fee', and 'frame'.
3. Set 'fee' to TRUE if the word evokes or triggers a FrameNet frame (e.g., verbs denoting actions, nouns denoting entities or concepts related to frames). Otherwise, set 'fee' to FALSE.
4. If 'fee' is TRUE, then for the parameter 'frame' either select an appropriate frame from FRAMES or set it to 'To_Review', which means there are no suitable frames in FRAMES. You can only use frames from FRAMES list.
5. If 'fee' is FALSE, set 'frame' to '-'.
6. Provide a single JSON array object with all 'word', 'fee', and 'frame' keys possessing their respective values.
7. Output only the JSON array object, no explanations or comments.
8. Do not revise and do not generate multiple answers.

EXAMPLE:
{example}

TEXT: {text}

FRAMES: {bfn_frames}
"""

In [ ]:
client = Groq(api_key="")

In [ ]:
responses = list()
all_sentences = list()
for text in ukwac:
    text = text.replace(',', '.')
    text = text.replace(';', '.')
    text = text.replace(':', '.')

    sentences = tokenizer.tokenize(text)
    for sent in sentences:
        all_sentences.append(sent)

        text_prompt = PROMPT.format(example=example,
                                    text=sent,
                                    bfn_frames=''.join(frames))
        completion = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[
                {
                    "role": "user",
                    "content": text_prompt
                }
            ],
            temperature=1,
            max_completion_tokens=1024,
            top_p=1,
            stream=True,
            stop=None,
        )

        response = str()
        for chunk in completion:
            response += chunk.choices[0].delta.content or ""
        responses.append(response)
        
        # break

In [ ]:
print(responses[0])

In [ ]:
len(all_sentences)

In [ ]:
len(responses)

In [ ]:
df_dict = {"sents": all_sentences[:1175], "preds": responses}
df = pd.DataFrame(df_dict)

df

In [ ]:
df.to_csv("en_ukwac_1_9_preds_subsentences.csv", index=False)

In [ ]:
with open("en_ukwac_0_first_sent_preds_2.txt", 'w') as f:
    f.write(responses[0])

In [ ]:
with open("en_ukwac_0_first_sent_preds.json", 'w') as f:
    f.write(json.dumps(responses[0], sort_keys=False, indent=4))